## Notebook tworzący CSV z podsumowaniem i wynikami pierwszego etapu, klasyfikacji binarnej na zbiorze MVTec

In [23]:
import pandas as pd
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent          
sys.path.append(str(BASE_DIR / "src"))
import utils

In [24]:
rows = [
    {"model": "ResNet18", "true_negative": 24, "false_positive": 30, "false_negative": 0, "true_positive": 18},
    {"model": "mobilenetv3", "true_negative": 51, "false_positive": 3, "false_negative": 3, "true_positive": 15},
    {"model": "efficientnetb0", "true_negative": 42, "false_positive": 12, "false_negative": 0, "true_positive": 18},
    {"model": "customcnn", "true_negative": 50, "false_positive":4, "false_negative": 2, "true_positive": 16}
]

In [25]:
df = pd.DataFrame(rows)

In [26]:
df

,model,true_negative,false_positive,false_negative,true_positive
0,ResNet18,24,30,0,18
1,mobilenetv3,51,3,3,15
2,efficientnetb0,42,12,0,18
3,customcnn,50,4,2,16


In [27]:
df['accuracy'] = (df['true_positive'] + df['true_negative'])/(df['true_negative']+ df['true_positive']+ df['false_positive'] + df['false_negative'])
df['recall_nok'] = df['true_positive']/(df['true_positive'] + df['false_negative'])
df['precision_nok'] = df['true_positive']/(df['true_positive'] + df['false_positive'])
df['F1_nok'] = 2 * df['precision_nok'] * df['recall_nok']/(df['precision_nok'] + df['recall_nok'])


In [28]:
info = {
    "ResNet18": dict(
        pretrained=True, lr='1e-3', num_epochs=30, patience=5, epochs_run=11, best_epoch=6,
        builder=utils.build_model_resnet18, ckpt="resnet18_baseline/best.pth"),
    "mobilenetv3": dict(
        pretrained=True, lr='1e-3', num_epochs=30, patience=5, epochs_run=24, best_epoch=19,
        builder=utils.build_model_mobilenetv3, ckpt="mobilenetv3/best.pth"),
    "efficientnetb0": dict(
        pretrained=True, lr='1e-3', num_epochs=30, patience=5, epochs_run=12, best_epoch=7,
        builder=utils.build_model_efficientnet_b0, ckpt="efficientnetb0/best.pth"),
    "customcnn": dict(
        pretrained=False, lr='1e-4', num_epochs=250, patience=250, epochs_run=250, best_epoch=237,
        builder=utils.build_model_custom_cnn, ckpt="custom_cnn/best_lr_4.pth"),
}

In [29]:
for name, v in info.items():
    net = v["builder"](2)
    v["n_params"] = sum(p.numel() for p in net.parameters())
    v["n_params_trainable"] = sum(p.numel() for p in net.parameters() if p.requires_grad)
    v["checkpoint_mb"] = round(Path(BASE_DIR, "models", v["ckpt"]).stat().st_size / 1e6, 2)


In [30]:

cols = ["pretrained", "lr", "num_epochs", "patience", "epochs_run", "best_epoch",
        "n_params", "n_params_trainable", "checkpoint_mb"]


In [31]:

for c in cols:
    df[c] = df["model"].map({name: v[c] for name, v in info.items()})

In [32]:
df

,model,true_negative,false_positive,false_negative,true_positive,accuracy,recall_nok,precision_nok,F1_nok,pretrained,lr,num_epochs,patience,epochs_run,best_epoch,n_params,n_params_trainable,checkpoint_mb
0,ResNet18,24,30,0,18,0.583333,1.000000,0.375000,0.545455,True,1e-3,30,5,11,6,11177538,11177538,44.78
1,mobilenetv3,51,3,3,15,0.916667,0.833333,0.833333,0.833333,True,1e-3,30,5,24,19,1519906,1519906,6.20
2,efficientnetb0,42,12,0,18,0.833333,1.000000,0.600000,0.750000,True,1e-3,30,5,12,7,4010110,4010110,16.31
3,customcnn,50,4,2,16,0.916667,0.888889,0.800000,0.842105,False,1e-4,250,250,250,237,97938,97938,0.40


In [33]:
df.to_csv(Path(BASE_DIR, 'results', 'first_stage_results.csv'), index=False)